# 🔬 离线调试：`agent/run` → `modelcraft_admin_agent`

**完全不走 server**：mock 掉 `make_client`（GraphQL 层），直接调 LangGraph。

复现请求：用户消息 `"列出我所有的项目"`，`layer=org`，`orgName=lukeco`

```bash
cd modelcraft-agent
jupyter notebook notebooks/debug_agent_run.ipynb
```

## 0. 环境初始化

In [ ]:
import sys, os, json, uuid
from unittest.mock import AsyncMock, patch, MagicMock

AGENT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if AGENT_ROOT not in sys.path:
    sys.path.insert(0, AGENT_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(AGENT_ROOT, '.env'))

from logging_setup import setup_logging
setup_logging()

import config
print('AGENT_ROOT:', AGENT_ROOT)
print('LLM_MODEL :', config.LLM_MODEL)

## 1. Mock GraphQL Client（核心：让工具不打网络）

所有 tool 都通过 `agents.shared.make_client(state)` 拿 GraphQL 客户端。  
只要 patch 这一个函数，全部工具调用就完全离线。

In [ ]:
# ── 修改这里来模拟不同数据场景 ──────────────────────────────────────────────
FAKE_PROJECTS = [
    {"id": "p1", "slug": "onboardceshi", "title": "onboard测试",  "description": "", "status": "ACTIVE"},
    {"id": "p2", "slug": "abcde",        "title": "abcde",        "description": "", "status": "ACTIVE"},
    {"id": "p3", "slug": "pro1",         "title": "pro1",         "description": "", "status": "ACTIVE"},
]


def make_fake_client(state=None):
    """返回一个假 GraphQL client，所有方法离线返回预设数据。"""
    c = MagicMock()
    c.list_projects    = AsyncMock(return_value={"data": {"projects": FAKE_PROJECTS}})
    c.list_databases   = AsyncMock(return_value={"data": {"listDatabases": {"edges": [
        {"node": {"name": "maindb"}}, {"node": {"name": "analyticsdb"}}
    ]}}})
    c.list_models      = AsyncMock(return_value={"data": {"models": {"items": [
        {"id": "m1", "name": "user",  "title": "用户"},
        {"id": "m2", "name": "order", "title": "订单"},
    ], "hasNextPage": False}}})
    c.get_model_fields = AsyncMock(return_value={"data": {"model": {"fields": []}}})
    c.query_model      = AsyncMock(return_value={"data": {"queryModel": {"items": [], "hasNextPage": False}}})
    c.nl2filter        = AsyncMock(return_value={"data": {"nl2filter": {"filter": "{}", "explanation": "mock"}}})
    return c


print('✅ make_fake_client 已定义')

## 2. 构造 AgentState（还原原始 `agent/run` 请求）

| AgentState 字段 | 来源 |
|---|---|
| `org_name` | `forwardedProps.orgName` |
| `layer` | `context[2].value.layer` |
| `project_slug` | `forwardedProps.projectSlug` |
| `authorization` | 离线填 fake token，不校验 |

In [ ]:
from langchain_core.messages import HumanMessage

# ── 对照原始 agent/run payload 修改 ──────────────────────────────────────────
USER_MESSAGE = "列出我所有的项目"   # body.messages[0].content
ORG_NAME     = "lukeco"             # forwardedProps.orgName
LAYER        = "org"                # context[2].value.layer
PROJECT_SLUG = "Default Project"    # forwardedProps.projectSlug
# ─────────────────────────────────────────────────────────────────────────────


def make_state(message: str = USER_MESSAGE) -> dict:
    return {
        "messages"     : [HumanMessage(content=message)],
        "authorization": "Bearer fake-offline-token",
        "org_name"     : ORG_NAME,
        "layer"        : LAYER,
        "project_slug" : PROJECT_SLUG,
        "current_model": "",
        "current_db"   : "",
    }


def new_cfg() -> dict:
    tid = str(uuid.uuid4())
    print('thread_id:', tid)
    return {"configurable": {"thread_id": tid}}

## 3. 运行 Admin Graph（离线 ainvoke）

In [ ]:
from agents.admin_agent import admin_graph

cfg = new_cfg()

with patch('agents.shared.make_client', side_effect=make_fake_client):
    result = await admin_graph.ainvoke(make_state(), config=cfg)

print('\n🤖 Agent 回复：')
print(result['messages'][-1].content)

## 4. Streaming（实时看 tool call → tool result → 回复）

In [ ]:
cfg = new_cfg()

with patch('agents.shared.make_client', side_effect=make_fake_client):
    async for event in admin_graph.astream_events(make_state(), config=cfg, version='v2'):
        kind = event['event']
        name = event.get('name', '')

        if kind == 'on_chat_model_stream':
            chunk = event['data']['chunk']
            if hasattr(chunk, 'content') and chunk.content:
                print(chunk.content, end='', flush=True)

        elif kind == 'on_tool_start':
            inp = event['data'].get('input', {})
            print(f'\n\n  ▶ [{name}]  args={json.dumps(inp, ensure_ascii=False)}')

        elif kind == 'on_tool_end':
            out = str(event['data'].get('output', ''))[:200]
            print(f'  ◀ [{name}]  → {out}')

print('\n\n[done]')

## 5. 场景切换

In [ ]:
# 场景 A：项目列表为空
empty = MagicMock()
empty.list_projects = AsyncMock(return_value={"data": {"projects": []}})

with patch('agents.shared.make_client', return_value=empty):
    r = await admin_graph.ainvoke(make_state(), config=new_cfg())

print('[空项目] Agent:', r['messages'][-1].content)

In [ ]:
# 场景 B：GraphQL 返回 error
err = MagicMock()
err.list_projects = AsyncMock(return_value={
    "errors": [{"message": "Unauthorized"}], "data": None
})

with patch('agents.shared.make_client', return_value=err):
    r = await admin_graph.ainvoke(make_state(), config=new_cfg())

print('[GraphQL error] Agent:', r['messages'][-1].content)

In [ ]:
# 场景 C：自定义消息
with patch('agents.shared.make_client', side_effect=make_fake_client):
    r = await admin_graph.ainvoke(make_state("帮我跳转到 abcde 项目"), config=new_cfg())

print('[跳转] Agent:', r['messages'][-1].content)

## 6. 检查完整消息链（tool_calls / ToolMessage 序列）

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage

with patch('agents.shared.make_client', side_effect=make_fake_client):
    result = await admin_graph.ainvoke(make_state(), config=new_cfg())

print(f'共 {len(result["messages"])} 条消息:\n')
for i, m in enumerate(result['messages']):
    role = type(m).__name__
    if isinstance(m, AIMessage) and getattr(m, 'tool_calls', None):
        calls = [f"{tc['name']}({json.dumps(tc['args'], ensure_ascii=False)})" for tc in m.tool_calls]
        print(f'  [{i}] {role}  tool_calls={calls}')
    elif isinstance(m, ToolMessage):
        print(f'  [{i}] {role}  id={m.tool_call_id}  → {str(m.content)[:80]}')
    else:
        print(f'  [{i}] {role}  {str(m.content)[:80]}')